In [1]:
%load_ext autoreload
%autoreload 2

In [20]:
import numpy as np
import popsim.param_utils as param_utils
from popsim.modules.tearing import DisruptionPhase, IslandRotationPhase, IslandModeNumber, Tearing
from popsim.simulate import simulate

dt = 1e-4 / 3  # s
time_base = param_utils.make_time_base(t0=0.0, t1=7.0, dt=dt)
config = Tearing.Config(
    magx_time=time_base, thincurr_file="21_mode_resp_data.txt", ods_file="thatfile.txt"
)


def find_nearest(array, value):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return array[idx]


def generate_disruption_phase_trajectory(trigger_time: float, tq_to_cq_dur: float):
    # TODO(allenw): we want a rectilinear interpolation scheme.
    disrupt_phase_dict = {
        0.0: DisruptionPhase.NONE,
        trigger_time - dt: DisruptionPhase.NONE,
        trigger_time: DisruptionPhase.TQ,
        trigger_time + tq_to_cq_dur - dt: DisruptionPhase.TQ,
        trigger_time + tq_to_cq_dur: DisruptionPhase.CQ,
    }

    # Round the times to the nearest time step in the time base.
    disrupt_phase_dict = {
        find_nearest(time_base, time): phase
        for time, phase in disrupt_phase_dict.items()
    }
    return disrupt_phase_dict

def generate_island_rotation_phase_trajectory(
    trigger_time: float, rot_dur: float, locking_dur: float
):
    # TODO(allenw): we want a rectilinear interpolation scheme.
    rot_phase_dict = {
        0.0: IslandRotationPhase.NONE,
        trigger_time - dt: IslandRotationPhase.NONE,
        trigger_time: IslandRotationPhase.SPAWN,
        trigger_time + dt: IslandRotationPhase.ROTATING,
        trigger_time + rot_dur - dt: IslandRotationPhase.ROTATING,
        trigger_time + rot_dur: IslandRotationPhase.DECELERATING,
        trigger_time + rot_dur + locking_dur - dt: IslandRotationPhase.DECELERATING,
        trigger_time + rot_dur + locking_dur: IslandRotationPhase.LOCKED,
    }

    # Round the times to the nearest time step in the time base.
    rot_phase_dict = {
        find_nearest(time_base, time): phase for time, phase in rot_phase_dict.items()
    }
    return rot_phase_dict

W = {mode: 0.0 for mode in IslandModeNumber}
F = {mode: 0.0 for mode in IslandModeNumber}
wave_phase={mode: 0.0 for mode in IslandModeNumber}

initial_state = Tearing.State(W=W, F=F, wave_phase=wave_phase)

q2_rot_freq = 7e3
rot_dur = 1.0
trigger_time = 5.0
disrupt_time = 6.5
dur_tq_to_spike = 1e-3
dur_cq = 10e-3
survival_time = 0.3
locking_dur = 0.2

params = Tearing.Params(
    q2_rot_freq=7e3,  # Hz
    rot_dur=1.0,  # s
    locking_dur=locking_dur,  # s
    disruption_phase=generate_disruption_phase_trajectory(
        disrupt_time, dur_tq_to_spike
    ),
    island_rotation_phase=generate_island_rotation_phase_trajectory(
        trigger_time, rot_dur, locking_dur
    ),
)

In [22]:
import jax
jax.config.update("jax_platforms", "cpu")
tearing_module = Tearing(config=config)

sol_xarray = simulate(tearing_module, time_base, initial_state, params)

In [23]:
from popsim.visualize import visualize_time_series

sol_xarray.to_netcdf("tearing_simulation.nc")
visualize_time_series(sol_xarray, max_cols=2)


BokehModel(combine_events=True, render_bundle={'docs_json': {'dd4cc014-301a-47a6-84ff-bda029146ff3': {'version…

In [32]:
import xarray as xr
from popsim.modules.tearing import IslandModeNumber
# Get the measured magnetic signal for each b-dot
curPerW = 1e3/1e-2 # 1 kA/cm <- a guess for now

# Convert island width to current
magnetic_measurements = xr.Dataset()
magnetic_measurements['time'] = sol_xarray.time

sensor_angles = [0, 45, 90, 135, 180, 225, 270, 315]
sensor_calibrations = [1, 1, 1, 1, 1, 1, 1, 1]
poloidal_sensor_measures = []
radial_sensor_measures = []
for i, angle in enumerate(sensor_angles):
    poloidal_sensor_measure = 0
    radial_sensor_measure = 0
    for mode in IslandModeNumber:
        mode_width = sol_xarray[f"state.W.{str(mode)}"]
        mode_angle = sol_xarray[f"state.wave_phase.{str(mode)}"]
        mode_current = mode_width * curPerW
        poloidal_sensor_measure += mode_current*sensor_calibrations[i]*np.cos(sensor_angles[i] - mode_angle)
        radial_sensor_measure += mode_current*sensor_calibrations[i]*np.sin(sensor_angles[i] - mode_angle)
    
    poloidal_sensor_measures.append(poloidal_sensor_measure)
    radial_sensor_measures.append(radial_sensor_measure)

magnetic_measurements['poloidal'] = xr.DataArray(poloidal_sensor_measures, dims=['sensor', 'time'])
magnetic_measurements['radial'] = xr.DataArray(radial_sensor_measures, dims=['sensor', 'time'])

magnetic_measurements.to_netcdf("magnetic_measurements.nc")
